In [ ]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
from dask.distributed import Client, LocalCluster, as_completed, get_client # Import get_client
from dask.distributed import fire_and_forget
from tqdm.auto import tqdm # For TQDM in a notebook/script environment
from typing import Dict, Any, List, Tuple
import os
import shutil
import random
import time

# --- Dask Client Setup ---
# It's good practice to ensure no lingering clusters, though Dask tries to find an available port.
# If you get "Port 8787 is already in use", it means a previous session might not have fully shut down.
# Restarting your kernel often resolves this, or just letting Dask pick a new port.
cluster = LocalCluster(n_workers=4, threads_per_worker=1) # Adjust workers as needed
client = Client(cluster)
print(f"Dask dashboard link: {client.dashboard_link}")

# --- Global Parameters ---
START_YEAR = 2025
END_YEAR = 2030
ITERATIONS = 5

# --- Utility Functions ---

def prepare_dask_params(params: Dict[str, Any]) -> Dict[str, Any]:
    """Prepares parameters for Dask, handling any special types if needed."""
    # For now, simply return the dictionary as is.
    # In more complex scenarios, you might convert NumPy arrays to Dask arrays, etc.
    return params

class TqdmCallback:
    """A Dask callback to integrate with tqdm for progress tracking."""
    def __init__(self, total, desc="Dask Computation"):
        self.total = total
        self.desc = desc
        self.pbar = None
        self.completed_tasks = 0

    def _start(self, dsk):
        self.pbar = tqdm(total=self.total, desc=self.desc)

    def _inc_progress(self, dsk, state, failed):
        newly_finished = len(state["finished"]) - self.completed_tasks
        self.pbar.update(newly_finished)
        self.completed_tasks = len(state["finished"])

    def _finish(self, dsk, state, failed):
        if self.pbar:
            self.pbar.close()

    def __enter__(self):
        self.pbar = tqdm(total=self.total, desc=self.desc)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.pbar:
            self.pbar.close()


# --- Data Generation Functions ---

def gen_population_raw_pandas(pop_size: int, year: int, seed: int, age_dist: np.ndarray, base_meta_df_for_dtypes: pd.DataFrame) -> pd.DataFrame:
    """
    Generates a raw Pandas DataFrame representing the initial population for a given year.
    This function is designed to be run within Dask workers.

    Args:
        pop_size (int): The size of the population to generate.
        year (int): The current simulation year.
        seed (int): A seed for reproducibility.
        age_dist (np.ndarray): Array of age distributions to sample from.
        base_meta_df_for_dtypes (pd.DataFrame): A DataFrame with at least one dummy row
                                                to reliably infer column dtypes.

    Returns:
        pd.DataFrame: A Pandas DataFrame with the initial population data.
    """
    np.random.seed(seed)
    random.seed(seed)

    df = pd.DataFrame({'person_id': np.arange(pop_size)})

    # Initialize core demographic columns
    df['sex'] = np.random.randint(0, 2, pop_size).astype('int64') # 0: Male, 1: Female
    df['age'] = np.random.choice(age_dist, size=pop_size).astype('int64')
    df['year_born'] = year - df['age']
    df['alive'] = True
    df['LTC'] = False # Long-Term Condition, initialize to False
    df['QALY'] = 0.0 # Quality-Adjusted Life Years
    df['cost'] = 0.0 # Healthcare costs
    df['school'] = np.where(df['age'] < 5, -1, np.where(df['age'] < 12, 0, 1)).astype('int64')

    # Initialize missing columns from base_meta_df_for_dtypes
    # base_meta_df_for_dtypes is guaranteed to have at least one row, so .iloc[0] is safe.
    for col in base_meta_df_for_dtypes.columns.drop(df.columns, errors='ignore'):
        col_dtype = base_meta_df_for_dtypes[col].dtype
        if pd.api.types.is_numeric_dtype(col_dtype):
            df[col] = 0 # Default to 0 for numeric types
        elif pd.api.types.is_bool_dtype(col_dtype):
            df[col] = False # Default to False for boolean types
        else:
            # For object or other types, you might want a specific default or pd.NA
            df[col] = base_meta_df_for_dtypes[col].iloc[0] # Use the dummy row value
        df[col] = df[col].astype(col_dtype) # Ensure the correct dtype is applied

    return df

def apply_mortality(partition: pd.DataFrame, abs_mort: np.ndarray, current_year: int) -> pd.DataFrame:
    """Applies mortality rates to a population partition."""
    if partition.empty:
        return partition

    # Ensure 'age' is int for indexing
    partition['age'] = partition['age'].astype(int)

    # Use .loc for safe indexing and to avoid SettingWithCopyWarning
    # Filter for alive individuals before applying mortality
    alive_mask = partition['alive']

    # Get mortality rates for current age and sex, ensure within bounds
    # abs_mort is indexed by age and then sex (e.g., abs_mort[age, sex])
    max_age_mort = abs_mort.shape[0] - 1
    safe_age = np.clip(partition.loc[alive_mask, 'age'], 0, max_age_mort)
    mortality_rates = abs_mort[safe_age, partition.loc[alive_mask, 'sex']]

    # Calculate probability of death
    # Random numbers for each person
    random_draws = np.random.rand(len(mortality_rates))
    death_mask = random_draws < mortality_rates

    # Apply death to the 'alive' column for individuals whose random_draws are less than mortality_rates
    partition.loc[alive_mask, 'alive'] = ~death_mask

    return partition

def update_age_and_year(partition: pd.DataFrame, current_year: int) -> pd.DataFrame:
    """Updates age for alive individuals and sets current year."""
    if partition.empty:
        return partition
    partition.loc[partition['alive'], 'age'] += 1
    # You might want a 'current_sim_year' column if needed for other logic
    # partition['current_sim_year'] = current_year
    return partition

def apply_births(partition: pd.DataFrame, current_year: int, fertility_rates_df: pd.DataFrame, seed_offset: int, base_meta_df_for_dtypes: pd.DataFrame) -> pd.DataFrame:
    """
    Applies births to a population partition.
    This version vectorizes birth generation for the whole partition.
    """
    if partition.empty:
        return partition

    female_population = partition[(partition['sex'] == 1) & (partition['alive'] == True)].copy()

    if female_population.empty:
        return partition

    # Merge fertility rates based on age. Assume fertility_rates_df has 'age' and 'rate' columns
    # Ensure 'age' is integer for merging/joining
    female_population['age_int'] = female_population['age'].astype(int)
    merged_pop = pd.merge(female_population, fertility_rates_df[['age', 'rate']],
                          left_on='age_int', right_on='age', how='left', suffixes=('_pop', '_fert'))
    merged_pop['rate'] = merged_pop['rate'].fillna(0) # Fill NaN rates with 0

    # Calculate expected number of births per woman
    merged_pop['expected_births'] = merged_pop['rate']

    # Sample actual number of births using Poisson distribution
    local_random_state = np.random.RandomState(seed_offset + hash(frozenset(partition.index.values.tobytes()))) # More robust seed per partition
    num_births_per_woman = local_random_state.poisson(merged_pop['expected_births'])

    # Create new births DataFrame
    total_new_births = num_births_per_woman.sum()
    if total_new_births > 0:
        new_born_ids = np.arange(partition['person_id'].max() + 1, partition['person_id'].max() + 1 + total_new_births)
        new_born_sexes = local_random_state.randint(0, 2, total_new_births) # 0: Male, 1: Female

        new_born_df = pd.DataFrame({
            'person_id': new_born_ids,
            'sex': new_born_sexes,
            'age': 0,
            'year_born': current_year,
            'alive': True,
            'LTC': False,
            'QALY': 0.0,
            'cost': 0.0,
            'school': -1 # Babies are pre-school age
        })

        # Add any other required columns, ensuring dtypes match the main population DataFrame
        # Use base_meta_df_for_dtypes to get correct dtypes for new columns
        for col in base_meta_df_for_dtypes.columns.drop(new_born_df.columns, errors='ignore'):
            col_dtype = base_meta_df_for_dtypes[col].dtype
            if pd.api.types.is_numeric_dtype(col_dtype):
                new_born_df[col] = 0
            elif pd.api.types.is_bool_dtype(col_dtype):
                new_born_df[col] = False
            else:
                new_born_df[col] = base_meta_df_for_dtypes[col].iloc[0] # Fallback to dummy row value
            new_born_df[col] = new_born_df[col].astype(col_dtype)


        partition = pd.concat([partition, new_born_df], ignore_index=True)

    return partition

def update_qaly_cost(partition: pd.DataFrame, current_year: int, life_exp_np: np.ndarray, emp_np: np.ndarray, morb_keys: Dict[str, Any]) -> pd.DataFrame:
    """Updates QALY and cost for the population based on current year's attributes."""
    if partition.empty:
        return partition

    # Only update for alive individuals
    alive_mask = partition['alive']
    alive_df = partition[alive_mask]

    if alive_df.empty:
        return partition

    # Example: QALY calculation (simplified)
    # Assume QALY depends on age and health state (LTC)
    # For demonstration, let's use a simplified age-based QALY decrement and LTC decrement
    base_qaly = 1.0 # Max QALY per year

    # Age-dependent QALY: For simplicity, let's say QALY decreases slightly with age
    # Ensure age is within bounds for life_exp_np if used for QALY
    max_age_qaly = life_exp_np.shape[0] - 1
    safe_age_qaly = np.clip(alive_df['age'], 0, max_age_qaly)
    # Using life_exp_np as a proxy for QALY reduction factor (e.g., higher life_exp, higher QALY)
    # This is a placeholder; actual QALY calculation would be more complex.
    qaly_reduction_factor = life_exp_np[safe_age_qaly, alive_df['sex']] / 80.0 # Normalize to something reasonable
    qaly_reduction_factor = np.clip(qaly_reduction_factor, 0.5, 1.0) # Keep between 0.5 and 1.0

    current_qaly = base_qaly * qaly_reduction_factor

    # LTC effect on QALY: If LTC is True, reduce QALY
    current_qaly = np.where(alive_df['LTC'], current_qaly * 0.7, current_qaly) # 30% QALY reduction for LTC

    # Ensure current_qaly has the same index as alive_df before assignment
    partition.loc[alive_mask, 'QALY'] += pd.Series(current_qaly, index=alive_df.index) # Accumulate QALYs annually

    # Example: Cost calculation (simplified)
    # Assume cost depends on age, LTC, and employment status (simplified to 'school' for now)
    base_cost_per_year = 1000 # Base healthcare cost

    # Age-dependent cost: Costs increase with age
    # Using emp_np as a proxy for cost factor (e.g., higher emp rate implies healthier, lower cost?)
    # This is a placeholder; actual cost calculation would be more complex.
    max_age_cost = emp_np.shape[0] - 1
    safe_age_cost = np.clip(alive_df['age'], 0, max_age_cost)
    cost_factor = 1.0 + (alive_df['age'] / 100.0) # Simple linear increase with age

    # LTC effect on cost: If LTC is True, increase cost
    cost_factor = np.where(alive_df['LTC'], cost_factor * 2.0, cost_factor) # Double cost for LTC

    current_cost = base_cost_per_year * cost_factor

    # Employment status effect (using 'school' as proxy for 'employed' or 'unemployed' states)
    # For example, if school == 1 (adult), assume higher engagement/cost
    # This is highly simplified and depends on your model's logic.
    current_cost = np.where(alive_df['school'] == 1, current_cost * 1.1, current_cost) # Adult population might have higher costs

    # Ensure current_cost has the same index as alive_df before assignment
    partition.loc[alive_mask, 'cost'] += pd.Series(current_cost, index=alive_df.index) # Accumulate costs annually

    return partition

def update_health_states(partition: pd.DataFrame, current_year: int, morb_keys: Dict[str, Any]) -> pd.DataFrame:
    """Updates health states (e.g., LTC status) for the population."""
    if partition.empty:
        return partition

    # Example: Simple probabilistic transition to LTC based on age
    # Placeholder: Assuming a base probability of developing LTC increases with age
    # This would typically involve specific transition matrices or rules from `morb_keys`

    # Only apply to alive individuals who don't already have LTC
    eligible_mask = (partition['alive'] == True) & (partition['LTC'] == False)
    eligible_df = partition[eligible_mask]

    if eligible_df.empty:
        return partition

    # Simplified probability of acquiring LTC
    # Let's say it's 0.001 + age * 0.0001
    prob_acquire_ltc = 0.001 + (eligible_df['age'] * morb_keys['dummy_morbidity_param_factor'])
    prob_acquire_ltc = np.clip(prob_acquire_ltc, 0, 0.1) # Cap max probability at 10%

    # Randomly determine if LTC is acquired
    random_draws = np.random.rand(len(eligible_df))
    acquire_ltc_mask = random_draws < prob_acquire_ltc

    partition.loc[eligible_mask, 'LTC'] = acquire_ltc_mask

    return partition

# --- Population System Class (Dask-enabled) ---

class PopulationSystemDask:
    def __init__(self, initial_pop_size: int, params: Dict[str, Any], scenario_type: str,
                 seed_offset: int, n_partitions: int, persist_dir: str = None):
        # Retrieve client within the worker context for this class's methods
        client = get_client()

        self.params = params
        self.scenario_type = scenario_type
        self.n_partitions = n_partitions
        self.persist_dir = persist_dir if persist_dir else f'dask-persist-{os.getpid()}-{time.time_ns()}'
        self.seed_offset = seed_offset

        # Retrieve published datasets from within the worker
        self.abs_mort = client.get_dataset("abs_mort_dataset")
        self.life_exp_np = client.get_dataset("life_exp_np_dataset")
        self.emp_np = client.get_dataset("emp_np_dataset")
        self.morb_keys = client.get_dataset("morb_keys_dataset")
        self.age_dist = client.get_dataset("age_dist_dataset")
        self.fertility_rates = client.get_dataset("fertility_rates_dataset")
        self.base_meta_df_for_dtypes = client.get_dataset("base_meta_df_for_dtypes_dataset")

        # The Dask metadata for the population DataFrame
        # This is derived from the base_meta_df_for_dtypes
        self.pop_meta = self.base_meta_df_for_dtypes.head(0)

        # Initial population generation
        initial_pop_dd = dd.from_delayed(
            client.submit(gen_population_raw_pandas, initial_pop_size, START_YEAR, self.seed_offset,
                          self.age_dist, self.base_meta_df_for_dtypes),
            meta=self.pop_meta
        ).repartition(npartitions=self.n_partitions)
        self.pop = initial_pop_dd.persist()

        # Setup persistence directory
        if self.persist_dir:
            os.makedirs(self.persist_dir, exist_ok=True)

    def run_year(self, year: int):
        """Runs the simulation for a single year."""
        # Retrieve client within the worker context for this method
        client = get_client()

        # 1. Update Age
        self.pop = self.pop.map_partitions(update_age_and_year, current_year=year, meta=self.pop_meta)
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

        # 2. Apply Mortality
        self.pop = self.pop.map_partitions(
            apply_mortality,
            abs_mort=self.abs_mort,
            current_year=year,
            meta=self.pop_meta
        )
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

        # 3. Apply Births
        self.pop = self.pop.map_partitions(
            apply_births,
            current_year=year,
            fertility_rates_df=self.fertility_rates,
            seed_offset=self.seed_offset + year * 1000,
            base_meta_df_for_dtypes=self.base_meta_df_for_dtypes,
            meta=self.pop_meta
        )
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

        # 4. Update QALY and Cost (for alive individuals)
        self.pop = self.pop.map_partitions(
            update_qaly_cost,
            current_year=year,
            life_exp_np=self.life_exp_np,
            emp_np=self.emp_np,
            morb_keys=self.morb_keys,
            meta=self.pop_meta
        )
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

        # 5. Update Health States (e.g., LTC for alive individuals)
        self.pop = self.pop.map_partitions(
            update_health_states,
            current_year=year,
            morb_keys=self.morb_keys,
            meta=self.pop_meta
        )
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

    def get_results(self) -> Tuple[float, float]:
        """Calculates total costs and QALYs for the simulation."""
        # Ensure 'alive' column is boolean for filtering
        self.pop['alive'] = self.pop['alive'].astype(bool)

        total_cost = self.pop['cost'].sum().compute()
        total_qaly = self.pop['QALY'].sum().compute()
        return total_cost, total_qaly

    def cleanup(self):
        """Cleans up any temporary persistence directories."""
        # Retrieve client within the worker context for this method
        client = get_client()
        if self.persist_dir and os.path.exists(self.persist_dir):
            shutil.rmtree(self.persist_dir)


# --- Simulation Runner Function ---

def run_sim(pop_size: int, sampled_params: Dict[str, Any], seed_idx: int,
            n_partitions_per_sim: int) -> Tuple[float, float]:
    """
    Runs a single simulation comparing retrospective and prospective arms for total costs and QALYs.

    Args:
        pop_size (int): Initial population size for this simulation.
        sampled_params (Dict[str, Any]): Parameters for this specific simulation run.
        seed_idx (int): Base seed for this simulation run to ensure reproducibility.
        n_partitions_per_sim (int): Number of Dask partitions for this simulation's population.

    Returns:
        Tuple[float, float]: A tuple containing (incremental_cost, incremental_qaly).
    """
    # Get the worker-local client
    worker_client = get_client()

    # Create a temporary directory for each simulation run
    temp_persist_dir = f'dask_sim_data_{os.getpid()}_{time.time_ns()}_{seed_idx}'

    # Initialize simulation systems for retrospective and prospective arms
    # All static global data (abs_mort, life_exp_np, etc.) are now retrieved within
    # PopulationSystemDask's __init__ method using get_client().get_dataset().
    # Thus, no need to pass them as arguments to PopulationSystemDask constructor here.
    sim_r = PopulationSystemDask(pop_size, sampled_params, 'retrospective',
                                 seed_offset=seed_idx, n_partitions=n_partitions_per_sim,
                                 persist_dir=os.path.join(temp_persist_dir, 'retrospective'))
    sim_p = PopulationSystemDask(pop_size, sampled_params, 'prospective',
                                 seed_offset=seed_idx + 100000, n_partitions=n_partitions_per_sim,
                                 persist_dir=os.path.join(temp_persist_dir, 'prospective'))

    # Run simulation for each year
    for year in range(START_YEAR, END_YEAR + 1):
        sim_r.run_year(year)
        sim_p.run_year(year)

    # Get results
    total_cost_r, total_qaly_r = sim_r.get_results()
    total_cost_p, total_qaly_p = sim_p.get_results()

    # Clean up persisted data using fire_and_forget with the worker_client
    fire_and_forget(worker_client.submit(sim_r.cleanup))
    fire_and_forget(worker_client.submit(sim_p.cleanup))
    # Also clean up the parent temporary directory
    fire_and_forget(worker_client.submit(shutil.rmtree, temp_persist_dir, ignore_errors=True)) # Added ignore_errors for robustness

    # Calculate incremental costs and QALYs
    incremental_cost = total_cost_p - total_cost_r
    incremental_qaly = total_qaly_p - total_qaly_r

    return incremental_cost, incremental_qaly


# --- Main Execution ---

if __name__ == "__main__":
    print(f"--- Running Module 6: Deterministic Scenarios ---")

    # --- 1. Define Global Static Data ---
    # These are placeholder arrays; replace with your actual data.
    ABS_MORT = np.random.rand(101, 2) * 0.05 # Age 0-100, 2 sexes
    LIFE_EXP_NP = np.random.rand(101, 2) * 80 # Age 0-100, 2 sexes
    EMP_NP = np.random.rand(101, 2) * 0.8 # Age 0-100, 2 sexes (employment rates)
    MORB_KEYS = {'dummy_morbidity_param_factor': 0.0001} # Placeholder

    AGE_DIST = np.random.normal(loc=30, scale=15, size=1000).astype(int)
    AGE_DIST = AGE_DIST[ (AGE_DIST >= 0) & (AGE_DIST <= 100) ] # Ensure ages are realistic
    AGE_DIST = np.unique(AGE_DIST) # Just unique ages for sampling, or use weighted distribution

    # Fertility rates: age-specific rates (example data)
    FERTILITY_RATES_DATA = {
        'age': np.arange(15, 50),
        'rate': np.linspace(0.005, 0.1, 35) # Example: rates increasing then decreasing
    }
    FERTILITY_RATES_DATA['rate'][17:25] *= 1.5 # Peak around 32-40
    FERTILITY_RATES_DATA['rate'][25:] *= np.linspace(1.0, 0.1, 10) # Decrease after peak

    FERTILITY_RATES_DF = pd.DataFrame(FERTILITY_RATES_DATA)

    # --- Create the base_meta_df_for_dtypes with a dummy row ---
    # This DataFrame will have a single row with default values for each column,
    # ensuring that .iloc[0] is always safe and provides the correct dtype.
    meta_cols_definition = {
        'person_id': 'int64',
        'sex': 'int64',
        'age': 'int64',
        'year_born': 'int64',
        'alive': 'bool',
        'LTC': 'bool',
        'QALY': 'float64',
        'cost': 'float64',
        'school': 'int64',
        # Add any other columns your simulation might introduce
        # 'health_status': 'category',
        # 'region': 'int64',
    }
    dummy_data_for_meta = {
        col: pd.NA if dtype == 'object' else (0 if 'int' in dtype or 'float' in dtype else False)
        for col, dtype in meta_cols_definition.items()
    }
    # Create the DataFrame with one dummy row and ensure correct dtypes
    BASE_META_DF_FOR_DTYPES = pd.DataFrame([dummy_data_for_meta]).astype(meta_cols_definition)
    print("Base metadata DataFrame for dtypes created with a dummy row.")


    # --- Publish all static data as datasets ---
    # These will be retrieved directly by workers using client.get_dataset()
    client.publish_dataset(abs_mort_dataset=ABS_MORT)
    client.publish_dataset(life_exp_np_dataset=LIFE_EXP_NP)
    client.publish_dataset(emp_np_dataset=EMP_NP)
    client.publish_dataset(morb_keys_dataset=MORB_KEYS)
    client.publish_dataset(age_dist_dataset=AGE_DIST)
    client.publish_dataset(fertility_rates_dataset=FERTILITY_RATES_DF)
    client.publish_dataset(base_meta_df_for_dtypes_dataset=BASE_META_DF_FOR_DTYPES)

    print("All global parameters published to Dask cluster as datasets.")

    # --- 2. Define Simulation Parameters for each run ---
    params = []
    pop_sizes = []
    seeds = []
    parts = []

    for i in range(ITERATIONS):
        params.append({'param_a': i * 0.1, 'param_b': 100 + i * 5})
        pop_sizes.append(100_000) # Initial population size for each run
        seeds.append(i * 1000) # Unique seed for each run
        parts.append(4) # Number of partitions for the Dask DataFrame in each simulation

    # --- 3. Run Simulations in Parallel ---
    # `client.map` now only receives iterables that vary per task.
    # All static global data is fetched inside `run_sim` using `client.get_dataset()`.
    # The `pure=False` is essential because `run_sim` interacts with the Dask cluster
    # (via `get_client().get_dataset()` and `fire_and_forget(get_client().submit(...))`).
    futures = client.map(run_sim,
                         pop_sizes,
                         params,
                         seeds,
                         parts,
                         pure=False
                        )

    # Use TqdmCallback to show progress
    with TqdmCallback(total=len(futures), desc="Deterministic Scenarios") as cb:
        deterministic_results_raw = client.gather(futures)

    deterministic_results: List[Dict[str, Any]] = []
    for i, (dc, dq) in enumerate(deterministic_results_raw):
        result = {
            'iteration': i,
            'param_a': params[i]['param_a'],
            'param_b': params[i]['param_b'],
            'incremental_cost': dc,
            'incremental_qaly': dq
        }
        deterministic_results.append(result)

    print("\n--- Simulation Results ---")
    for res in deterministic_results:
        print(f"Iteration {res['iteration']}: Cost Diff={res['incremental_cost']:.2f}, QALY Diff={res['incremental_qaly']:.2f}")

    # --- Shutdown Dask Client and Cluster ---
    client.close()
    cluster.close()
    print("\nDask client and cluster shut down.")
    

Dask dashboard link: http://127.0.0.1:8787/status
--- Running Module 6: Deterministic Scenarios ---
Base metadata DataFrame for dtypes created with a dummy row.
All global parameters published to Dask cluster as datasets.


Deterministic Scenarios:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
from dask.distributed import Client, LocalCluster, get_client # Removed as_completed
from dask.distributed import fire_and_forget
from tqdm.auto import tqdm
from typing import Dict, Any, List, Tuple
import os
import shutil
import random
import time

# --- Dask Client Setup ---
# It's good practice to ensure no lingering clusters, though Dask tries to find an available port.
# If you get "Port 8787 is already in use", it means a previous session might not have fully shut down.
# Restarting your kernel often resolves this, or just letting Dask pick a new port.
cluster = LocalCluster(n_workers=4, threads_per_worker=1) # Adjust workers as needed
client = Client(cluster)
print(f"Dask dashboard link: {client.dashboard_link}")

# --- Global Parameters ---
START_YEAR = 2025
END_YEAR = 2030
ITERATIONS = 5

# --- Utility Functions ---

# prepare_dask_params is no longer needed as we're using client.publish_dataset and get_client().get_dataset()

class TqdmCallback:
    """A Dask callback to integrate with tqdm for progress tracking."""
    def __init__(self, total, desc="Dask Computation"):
        self.total = total
        self.desc = desc
        self.pbar = None
        self.completed_tasks = 0

    def _start(self, dsk):
        self.pbar = tqdm(total=self.total, desc=self.desc)

    def _inc_progress(self, dsk, state, failed):
        newly_finished = len(state["finished"]) - self.completed_tasks
        self.pbar.update(newly_finished)
        self.completed_tasks = len(state["finished"])

    def _finish(self, dsk, state, failed):
        if self.pbar:
            self.pbar.close()

    def __enter__(self):
        self.pbar = tqdm(total=self.total, desc=self.desc)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.pbar:
            self.pbar.close()


# --- Data Generation Functions ---

def gen_population_raw_pandas(pop_size: int, year: int, seed: int, age_dist: np.ndarray, base_meta_df_for_dtypes: pd.DataFrame) -> pd.DataFrame:
    """
    Generates a raw Pandas DataFrame representing the initial population for a given year.
    This function is designed to be run within Dask workers.

    Args:
        pop_size (int): The size of the population to generate.
        year (int): The current simulation year.
        seed (int): A seed for reproducibility.
        age_dist (np.ndarray): Array of age distributions to sample from.
        base_meta_df_for_dtypes (pd.DataFrame): A DataFrame with at least one dummy row
                                                to reliably infer column dtypes.

    Returns:
        pd.DataFrame: A Pandas DataFrame with the initial population data.
    """
    np.random.seed(seed)
    random.seed(seed)

    df = pd.DataFrame({'person_id': np.arange(pop_size)})

    # Initialize core demographic columns
    df['sex'] = np.random.randint(0, 2, pop_size).astype('int64') # 0: Male, 1: Female
    df['age'] = np.random.choice(age_dist, size=pop_size).astype('int64')
    df['year_born'] = year - df['age']
    df['alive'] = True
    df['LTC'] = False # Long-Term Condition, initialize to False
    df['QALY'] = 0.0 # Quality-Adjusted Life Years
    df['cost'] = 0.0 # Healthcare costs
    df['school'] = np.where(df['age'] < 5, -1, np.where(df['age'] < 12, 0, 1)).astype('int64')

    # Initialize missing columns from base_meta_df_for_dtypes
    # base_meta_df_for_dtypes is guaranteed to have at least one row, so .iloc[0] is safe.
    for col in base_meta_df_for_dtypes.columns.drop(df.columns, errors='ignore'):
        col_dtype = base_meta_df_for_dtypes[col].dtype
        if pd.api.types.is_numeric_dtype(col_dtype):
            df[col] = 0 # Default to 0 for numeric types
        elif pd.api.types.is_bool_dtype(col_dtype):
            df[col] = False # Default to False for boolean types
        else:
            # For object or other types, you might want a specific default or pd.NA
            df[col] = base_meta_df_for_dtypes[col].iloc[0] # Use the dummy row value
        df[col] = df[col].astype(col_dtype) # Ensure the correct dtype is applied

    return df

def apply_mortality(partition: pd.DataFrame, abs_mort: np.ndarray, current_year: int) -> pd.DataFrame:
    """Applies mortality rates to a population partition."""
    if partition.empty:
        return partition

    # Ensure 'age' is int for indexing
    partition['age'] = partition['age'].astype(int)

    # Use .loc for safe indexing and to avoid SettingWithCopyWarning
    # Filter for alive individuals before applying mortality
    alive_mask = partition['alive']

    # Get mortality rates for current age and sex, ensure within bounds
    # abs_mort is indexed by age and then sex (e.g., abs_mort[age, sex])
    max_age_mort = abs_mort.shape[0] - 1
    safe_age = np.clip(partition.loc[alive_mask, 'age'], 0, max_age_mort)
    mortality_rates = abs_mort[safe_age, partition.loc[alive_mask, 'sex']]

    # Calculate probability of death
    # Random numbers for each person
    random_draws = np.random.rand(len(mortality_rates))
    death_mask = random_draws < mortality_rates

    # Apply death to the 'alive' column for individuals whose random_draws are less than mortality_rates
    partition.loc[alive_mask, 'alive'] = ~death_mask

    return partition

def update_age_and_year(partition: pd.DataFrame, current_year: int) -> pd.DataFrame:
    """Updates age for alive individuals and sets current year."""
    if partition.empty:
        return partition
    partition.loc[partition['alive'], 'age'] += 1
    # You might want a 'current_sim_year' column if needed for other logic
    # partition['current_sim_year'] = current_year
    return partition

def apply_births(partition: pd.DataFrame, current_year: int, fertility_rates_df: pd.DataFrame, seed_offset: int, base_meta_df_for_dtypes: pd.DataFrame) -> pd.DataFrame:
    """
    Applies births to a population partition.
    This version vectorizes birth generation for the whole partition.
    """
    if partition.empty:
        return partition

    female_population = partition[(partition['sex'] == 1) & (partition['alive'] == True)].copy()

    if female_population.empty:
        return partition

    # Merge fertility rates based on age. Assume fertility_rates_df has 'age' and 'rate' columns
    # Ensure 'age' is integer for merging/joining
    female_population['age_int'] = female_population['age'].astype(int)
    merged_pop = pd.merge(female_population, fertility_rates_df[['age', 'rate']],
                          left_on='age_int', right_on='age', how='left', suffixes=('_pop', '_fert'))
    merged_pop['rate'] = merged_pop['rate'].fillna(0) # Fill NaN rates with 0

    # Calculate expected number of births per woman
    merged_pop['expected_births'] = merged_pop['rate']

    # Sample actual number of births using Poisson distribution
    local_random_state = np.random.RandomState(seed_offset + hash(frozenset(partition.index.values.tobytes()))) # More robust seed per partition
    num_births_per_woman = local_random_state.poisson(merged_pop['expected_births'])

    # Create new births DataFrame
    total_new_births = num_births_per_woman.sum()
    if total_new_births > 0:
        new_born_ids = np.arange(partition['person_id'].max() + 1, partition['person_id'].max() + 1 + total_new_births)
        new_born_sexes = local_random_state.randint(0, 2, total_new_births) # 0: Male, 1: Female

        new_born_df = pd.DataFrame({
            'person_id': new_born_ids,
            'sex': new_born_sexes,
            'age': 0,
            'year_born': current_year,
            'alive': True,
            'LTC': False,
            'QALY': 0.0,
            'cost': 0.0,
            'school': -1 # Babies are pre-school age
        })

        # Add any other required columns, ensuring dtypes match the main population DataFrame
        # Use base_meta_df_for_dtypes to get correct dtypes for new columns
        for col in base_meta_df_for_dtypes.columns.drop(new_born_df.columns, errors='ignore'):
            col_dtype = base_meta_df_for_dtypes[col].dtype
            if pd.api.types.is_numeric_dtype(col_dtype):
                new_born_df[col] = 0
            elif pd.api.types.is_bool_dtype(col_dtype):
                new_born_df[col] = False
            else:
                new_born_df[col] = base_meta_df_for_dtypes[col].iloc[0] # Fallback to dummy row value
            new_born_df[col] = new_born_df[col].astype(col_dtype)


        partition = pd.concat([partition, new_born_df], ignore_index=True)

    return partition

def update_qaly_cost(partition: pd.DataFrame, current_year: int, life_exp_np: np.ndarray, emp_np: np.ndarray, morb_keys: Dict[str, Any]) -> pd.DataFrame:
    """Updates QALY and cost for the population based on current year's attributes."""
    if partition.empty:
        return partition

    # Only update for alive individuals
    alive_mask = partition['alive']
    alive_df = partition[alive_mask]

    if alive_df.empty:
        return partition

    # Example: QALY calculation (simplified)
    # Assume QALY depends on age and health state (LTC)
    # For demonstration, let's use a simplified age-based QALY decrement and LTC decrement
    base_qaly = 1.0 # Max QALY per year

    # Age-dependent QALY: For simplicity, let's say QALY decreases slightly with age
    # Ensure age is within bounds for life_exp_np if used for QALY
    max_age_qaly = life_exp_np.shape[0] - 1
    safe_age_qaly = np.clip(alive_df['age'], 0, max_age_qaly)
    # Using life_exp_np as a proxy for QALY reduction factor (e.g., higher life_exp, higher QALY)
    # This is a placeholder; actual QALY calculation would be more complex.
    qaly_reduction_factor = life_exp_np[safe_age_qaly, alive_df['sex']] / 80.0 # Normalize to something reasonable
    qaly_reduction_factor = np.clip(qaly_reduction_factor, 0.5, 1.0) # Keep between 0.5 and 1.0

    current_qaly = base_qaly * qaly_reduction_factor

    # LTC effect on QALY: If LTC is True, reduce QALY
    current_qaly = np.where(alive_df['LTC'], current_qaly * 0.7, current_qaly) # 30% QALY reduction for LTC

    # Ensure current_qaly has the same index as alive_df before assignment
    partition.loc[alive_mask, 'QALY'] += pd.Series(current_qaly, index=alive_df.index) # Accumulate QALYs annually

    # Example: Cost calculation (simplified)
    # Assume cost depends on age, LTC, and employment status (simplified to 'school' for now)
    base_cost_per_year = 1000 # Base healthcare cost

    # Age-dependent cost: Costs increase with age
    # Using emp_np as a proxy for cost factor (e.g., higher emp rate implies healthier, lower cost?)
    # This is a placeholder; actual cost calculation would be more complex.
    max_age_cost = emp_np.shape[0] - 1
    safe_age_cost = np.clip(alive_df['age'], 0, max_age_cost)
    cost_factor = 1.0 + (alive_df['age'] / 100.0) # Simple linear increase with age

    # LTC effect on cost: If LTC is True, increase cost
    cost_factor = np.where(alive_df['LTC'], cost_factor * 2.0, cost_factor) # Double cost for LTC

    # Employment status effect (using 'school' as proxy for 'employed' or 'unemployed' states)
    # For example, if school == 1 (adult), assume higher engagement/cost
    # This is highly simplified and depends on your model's logic.
    current_cost = np.where(alive_df['school'] == 1, current_cost * 1.1, current_cost) # Adult population might have higher costs

    # Ensure current_cost has the same index as alive_df before assignment
    partition.loc[alive_mask, 'cost'] += pd.Series(current_cost, index=alive_df.index) # Accumulate costs annually

    return partition

def update_health_states(partition: pd.DataFrame, current_year: int, morb_keys: Dict[str, Any]) -> pd.DataFrame:
    """Updates health states (e.g., LTC status) for the population."""
    if partition.empty:
        return partition

    # Example: Simple probabilistic transition to LTC based on age
    # Placeholder: Assuming a base probability of developing LTC increases with age
    # This would typically involve specific transition matrices or rules from `morb_keys`

    # Only apply to alive individuals who don't already have LTC
    eligible_mask = (partition['alive'] == True) & (partition['LTC'] == False)
    eligible_df = partition[eligible_mask]

    if eligible_df.empty:
        return partition

    # Simplified probability of acquiring LTC
    # Let's say it's 0.001 + age * 0.0001
    prob_acquire_ltc = 0.001 + (eligible_df['age'] * morb_keys['dummy_morbidity_param_factor'])
    prob_acquire_ltc = np.clip(prob_acquire_ltc, 0, 0.1) # Cap max probability at 10%

    # Randomly determine if LTC is acquired
    random_draws = np.random.rand(len(eligible_df))
    acquire_ltc_mask = random_draws < prob_acquire_ltc

    partition.loc[eligible_mask, 'LTC'] = acquire_ltc_mask

    return partition

# --- Population System Class (Dask-enabled) ---

class PopulationSystemDask:
    def __init__(self, initial_pop_size: int, params: Dict[str, Any], scenario_type: str,
                 seed_offset: int, n_partitions: int, persist_dir: str = None):
        # Retrieve client within the worker context for this class's methods
        client = get_client()

        self.params = params
        self.scenario_type = scenario_type
        self.n_partitions = n_partitions
        self.persist_dir = persist_dir if persist_dir else f'dask-persist-{os.getpid()}-{time.time_ns()}'
        self.seed_offset = seed_offset

        # Retrieve published datasets from within the worker
        self.abs_mort = client.get_dataset("abs_mort_dataset")
        self.life_exp_np = client.get_dataset("life_exp_np_dataset")
        self.emp_np = client.get_dataset("emp_np_dataset")
        self.morb_keys = client.get_dataset("morb_keys_dataset")
        self.age_dist = client.get_dataset("age_dist_dataset")
        self.fertility_rates = client.get_dataset("fertility_rates_dataset")
        self.base_meta_df_for_dtypes = client.get_dataset("base_meta_df_for_dtypes_dataset")

        # The Dask metadata for the population DataFrame
        # This is derived from the base_meta_df_for_dtypes
        self.pop_meta = self.base_meta_df_for_dtypes.head(0)

        # Initial population generation
        initial_pop_dd = dd.from_delayed(
            client.submit(gen_population_raw_pandas, initial_pop_size, START_YEAR, self.seed_offset,
                          self.age_dist, self.base_meta_df_for_dtypes),
            meta=self.pop_meta
        ).repartition(npartitions=self.n_partitions)
        self.pop = initial_pop_dd.persist()

        # Setup persistence directory
        if self.persist_dir:
            os.makedirs(self.persist_dir, exist_ok=True)

    def run_year(self, year: int):
        """Runs the simulation for a single year."""
        # Retrieve client within the worker context for this method
        client = get_client()

        # 1. Update Age
        self.pop = self.pop.map_partitions(update_age_and_year, current_year=year, meta=self.pop_meta)
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

        # 2. Apply Mortality
        self.pop = self.pop.map_partitions(
            apply_mortality,
            abs_mort=self.abs_mort,
            current_year=year,
            meta=self.pop_meta
        )
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

        # 3. Apply Births
        self.pop = self.pop.map_partitions(
            apply_births,
            current_year=year,
            fertility_rates_df=self.fertility_rates,
            seed_offset=self.seed_offset + year * 1000,
            base_meta_df_for_dtypes=self.base_meta_df_for_dtypes,
            meta=self.pop_meta
        )
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

        # 4. Update QALY and Cost (for alive individuals)
        self.pop = self.pop.map_partitions(
            update_qaly_cost,
            current_year=year,
            life_exp_np=self.life_exp_np,
            emp_np=self.emp_np,
            morb_keys=self.morb_keys,
            meta=self.pop_meta
        )
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

        # 5. Update Health States (e.g., LTC for alive individuals)
        self.pop = self.pop.map_partitions(
            update_health_states,
            current_year=year,
            morb_keys=self.morb_keys,
            meta=self.pop_meta
        )
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

    def get_results(self) -> Tuple[float, float]:
        """Calculates total costs and QALYs for the simulation."""
        # Ensure 'alive' column is boolean for filtering
        self.pop['alive'] = self.pop['alive'].astype(bool)

        total_cost = self.pop['cost'].sum().compute()
        total_qaly = self.pop['QALY'].sum().compute()
        return total_cost, total_qaly

    def cleanup(self):
        """Cleans up any temporary persistence directories."""
        # Retrieve client within the worker context for this method
        client = get_client() # Unused, but good to show consistency for distributed calls
        if self.persist_dir and os.path.exists(self.persist_dir):
            shutil.rmtree(self.persist_dir)


# --- Simulation Runner Function ---

def run_sim(pop_size: int, sampled_params: Dict[str, Any], seed_idx: int,
            n_partitions_per_sim: int) -> Tuple[float, float]:
    """
    Runs a single simulation comparing retrospective and prospective arms for total costs and QALYs.

    Args:
        pop_size (int): Initial population size for this simulation.
        sampled_params (Dict[str, Any]): Parameters for this specific simulation run.
        seed_idx (int): Base seed for this simulation run to ensure reproducibility.
        n_partitions_per_sim (int): Number of Dask partitions for this simulation's population.

    Returns:
        Tuple[float, float]: A tuple containing (incremental_cost, incremental_qaly).
    """
    # Get the worker-local client
    worker_client = get_client()

    # Create a temporary directory for each simulation run
    temp_persist_dir = f'dask_sim_data_{os.getpid()}_{time.time_ns()}_{seed_idx}'

    # Initialize simulation systems for retrospective and prospective arms
    # All static global data (abs_mort, life_exp_np, etc.) are now retrieved within
    # PopulationSystemDask's __init__ method using get_client().get_dataset().
    # Thus, no need to pass them as arguments to PopulationSystemDask constructor here.
    sim_r = PopulationSystemDask(pop_size, sampled_params, 'retrospective',
                                 seed_offset=seed_idx, n_partitions=n_partitions_per_sim,
                                 persist_dir=os.path.join(temp_persist_dir, 'retrospective'))
    sim_p = PopulationSystemDask(pop_size, sampled_params, 'prospective',
                                 seed_offset=seed_idx + 100000, n_partitions=n_partitions_per_sim,
                                 persist_dir=os.path.join(temp_persist_dir, 'prospective'))

    # Run simulation for each year
    for year in range(START_YEAR, END_YEAR + 1):
        sim_r.run_year(year)
        sim_p.run_year(year)

    # Get results
    total_cost_r, total_qaly_r = sim_r.get_results()
    total_cost_p, total_qaly_p = sim_p.get_results()

    # Clean up persisted data using fire_and_forget with the worker_client
    fire_and_forget(worker_client.submit(sim_r.cleanup))
    fire_and_forget(worker_client.submit(sim_p.cleanup))
    # Also clean up the parent temporary directory
    fire_and_forget(worker_client.submit(shutil.rmtree, temp_persist_dir, ignore_errors=True)) # Added ignore_errors for robustness

    # Calculate incremental costs and QALYs
    incremental_cost = total_cost_p - total_cost_r
    incremental_qaly = total_qaly_p - total_qaly_r

    return incremental_cost, incremental_qaly


# --- Main Execution ---

if __name__ == "__main__":
    print(f"--- Running Module 6: Deterministic Scenarios ---")

    # --- 1. Define Global Static Data ---
    # These are placeholder arrays; replace with your actual data.
    ABS_MORT = np.random.rand(101, 2) * 0.05 # Age 0-100, 2 sexes
    LIFE_EXP_NP = np.random.rand(101, 2) * 80 # Age 0-100, 2 sexes
    EMP_NP = np.random.rand(101, 2) * 0.8 # Age 0-100, 2 sexes (employment rates)
    MORB_KEYS = {'dummy_morbidity_param_factor': 0.0001} # Placeholder

    AGE_DIST = np.random.normal(loc=30, scale=15, size=1000).astype(int)
    AGE_DIST = AGE_DIST[ (AGE_DIST >= 0) & (AGE_DIST <= 100) ] # Ensure ages are realistic
    AGE_DIST = np.unique(AGE_DIST) # Just unique ages for sampling, or use weighted distribution

    # Fertility rates: age-specific rates (example data)
    FERTILITY_RATES_DATA = {
        'age': np.arange(15, 50),
        'rate': np.linspace(0.005, 0.1, 35) # Example: rates increasing then decreasing
    }
    FERTILITY_RATES_DATA['rate'][17:25] *= 1.5 # Peak around 32-40
    FERTILITY_RATES_DATA['rate'][25:] *= np.linspace(1.0, 0.1, 10) # Decrease after peak

    FERTILITY_RATES_DF = pd.DataFrame(FERTILITY_RATES_DATA)

    # --- Create the base_meta_df_for_dtypes with a dummy row ---
    # This DataFrame will have a single row with default values for each column,
    # ensuring that .iloc[0] is always safe and provides the correct dtype.
    meta_cols_definition = {
        'person_id': 'int64',
        'sex': 'int64',
        'age': 'int64',
        'year_born': 'int64',
        'alive': 'bool',
        'LTC': 'bool',
        'QALY': 'float64',
        'cost': 'float64',
        'school': 'int64',
        # Add any other columns your simulation might introduce
        # 'health_status': 'category',
        # 'region': 'int64',
    }
    dummy_data_for_meta = {
        col: pd.NA if dtype == 'object' else (0 if 'int' in dtype or 'float' in dtype else False)
        for col, dtype in meta_cols_definition.items()
    }
    # Create the DataFrame with one dummy row and ensure correct dtypes
    BASE_META_DF_FOR_DTYPES = pd.DataFrame([dummy_data_for_meta]).astype(meta_cols_definition)
    print("Base metadata DataFrame for dtypes created with a dummy row.")


    # --- Publish all static data as datasets ---
    # These will be retrieved directly by workers using client.get_dataset()
    client.publish_dataset(abs_mort_dataset=ABS_MORT)
    client.publish_dataset(life_exp_np_dataset=LIFE_EXP_NP)
    client.publish_dataset(emp_np_dataset=EMP_NP)
    client.publish_dataset(morb_keys_dataset=MORB_KEYS)
    client.publish_dataset(age_dist_dataset=AGE_DIST)
    client.publish_dataset(fertility_rates_dataset=FERTILITY_RATES_DF)
    client.publish_dataset(base_meta_df_for_dtypes_dataset=BASE_META_DF_FOR_DTYPES)

    print("All global parameters published to Dask cluster as datasets.")

    # --- 2. Define Simulation Parameters for each run ---
    params = []
    pop_sizes = []
    seeds = []
    parts = []

    for i in range(ITERATIONS):
        params.append({'param_a': i * 0.1, 'param_b': 100 + i * 5})
        pop_sizes.append(100_000) # Initial population size for each run
        seeds.append(i * 1000) # Unique seed for each run
        parts.append(4) # Number of partitions for the Dask DataFrame in each simulation

    # --- 3. Run Simulations in Parallel ---
    # `client.map` now only receives iterables that vary per task.
    # All static global data is fetched inside `run_sim` using `get_client().get_dataset()`.
    # The `pure=False` is essential because `run_sim` interacts with the Dask cluster
    # (via `get_client().get_dataset()` and `fire_and_forget(get_client().submit(...))`).
    futures = client.map(run_sim,
                         pop_sizes,
                         params,
                         seeds,
                         parts,
                         pure=False
                        )

    # Use TqdmCallback to show progress
    with TqdmCallback(total=len(futures), desc="Deterministic Scenarios") as cb:
        deterministic_results_raw = client.gather(futures)

    deterministic_results: List[Dict[str, Any]] = []
    for i, (dc, dq) in enumerate(deterministic_results_raw):
        result = {
            'iteration': i,
            'param_a': params[i]['param_a'],
            'param_b': params[i]['param_b'],
            'incremental_cost': dc,
            'incremental_qaly': dq
        }
        deterministic_results.append(result)

    print("\n--- Simulation Results ---")
    for res in deterministic_results:
        print(f"Iteration {res['iteration']}: Cost Diff={res['incremental_cost']:.2f}, QALY Diff={res['incremental_qaly']:.2f}")

    # --- Shutdown Dask Client and Cluster ---
    client.close()
    cluster.close()
    print("\nDask client and cluster shut down.")

Dask dashboard link: http://127.0.0.1:8787/status
--- Running Module 6: Deterministic Scenarios ---
Base metadata DataFrame for dtypes created with a dummy row.
All global parameters published to Dask cluster as datasets.


Deterministic Scenarios:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
from dask.distributed import Client, LocalCluster, get_client
from dask.distributed import fire_and_forget
from tqdm.auto import tqdm
from typing import Dict, Any, List, Tuple
import os
import shutil
# Removed 'random' and 'time' imports as they are no longer directly used at the module level
# or in ways that require specific module-level import here outside of numpy's random.

# --- Dask Client Setup ---
# It's good practice to ensure no lingering clusters, though Dask tries to find an available port.
# If you get "Port 8787 is already in use", it means a previous session might not have fully shut down.
# Restarting your kernel often resolves this, or just letting Dask pick a new port.
cluster = LocalCluster(n_workers=4, threads_per_worker=1) # Adjust workers as needed
client = Client(cluster)
print(f"Dask dashboard link: {client.dashboard_link}")

# --- Global Parameters ---
START_YEAR = 2025
END_YEAR = 2030
ITERATIONS = 5

# --- Utility Functions ---

class TqdmCallback:
    """A Dask callback to integrate with tqdm for progress tracking."""
    def __init__(self, total, desc="Dask Computation"):
        self.total = total
        self.desc = desc
        self.pbar = None
        self.completed_tasks = 0

    def _start(self, dsk):
        self.pbar = tqdm(total=self.total, desc=self.desc)

    def _inc_progress(self, dsk, state, failed):
        newly_finished = len(state["finished"]) - self.completed_tasks
        self.pbar.update(newly_finished)
        self.completed_tasks = len(state["finished"])

    def _finish(self, dsk, state, failed):
        if self.pbar:
            self.pbar.close()

    def __enter__(self):
        self.pbar = tqdm(total=self.total, desc=self.desc)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.pbar:
            self.pbar.close()


# --- Data Generation Functions ---

def gen_population_raw_pandas(pop_size: int, year: int, seed: int, age_dist: np.ndarray, base_meta_df_for_dtypes: pd.DataFrame) -> pd.DataFrame:
    """
    Generates a raw Pandas DataFrame representing the initial population for a given year.
    This function is designed to be run within Dask workers.

    Args:
        pop_size (int): The size of the population to generate.
        year (int): The current simulation year.
        seed (int): A seed for reproducibility.
        age_dist (np.ndarray): Array of age distributions to sample from.
        base_meta_df_for_dtypes (pd.DataFrame): A DataFrame with at least one dummy row
                                                to reliably infer column dtypes.

    Returns:
        pd.DataFrame: A Pandas DataFrame with the initial population data.
    """
    np.random.seed(seed)
    # The 'random' module (Python's built-in) is not used here; numpy's random is sufficient
    # random.seed(seed) # Removed as it's redundant with np.random.seed for this function's scope

    df = pd.DataFrame({'person_id': np.arange(pop_size)})

    # Initialize core demographic columns
    df['sex'] = np.random.randint(0, 2, pop_size).astype('int64') # 0: Male, 1: Female
    df['age'] = np.random.choice(age_dist, size=pop_size).astype('int64')
    df['year_born'] = year - df['age']
    df['alive'] = True
    df['LTC'] = False # Long-Term Condition, initialize to False
    df['QALY'] = 0.0 # Quality-Adjusted Life Years
    df['cost'] = 0.0 # Healthcare costs
    df['school'] = np.where(df['age'] < 5, -1, np.where(df['age'] < 12, 0, 1)).astype('int64')

    # Initialize missing columns from base_meta_df_for_dtypes
    # base_meta_df_for_dtypes is guaranteed to have at least one row, so .iloc[0] is safe.
    for col in base_meta_df_for_dtypes.columns.drop(df.columns, errors='ignore'):
        col_dtype = base_meta_df_for_dtypes[col].dtype
        if pd.api.types.is_numeric_dtype(col_dtype):
            df[col] = 0 # Default to 0 for numeric types
        elif pd.api.types.is_bool_dtype(col_dtype):
            df[col] = False # Default to False for boolean types
        else:
            # For object or other types, you might want a specific default or pd.NA
            df[col] = base_meta_df_for_dtypes[col].iloc[0] # Use the dummy row value
        df[col] = df[col].astype(col_dtype) # Ensure the correct dtype is applied

    return df

def apply_mortality(partition: pd.DataFrame, abs_mort: np.ndarray, current_year: int) -> pd.DataFrame:
    """Applies mortality rates to a population partition."""
    if partition.empty:
        return partition

    # Ensure 'age' is int for indexing
    partition['age'] = partition['age'].astype(int)

    # Use .loc for safe indexing and to avoid SettingWithCopyWarning
    # Filter for alive individuals before applying mortality
    alive_mask = partition['alive']

    # Get mortality rates for current age and sex, ensure within bounds
    # abs_mort is indexed by age and then sex (e.g., abs_mort[age, sex])
    max_age_mort = abs_mort.shape[0] - 1
    safe_age = np.clip(partition.loc[alive_mask, 'age'], 0, max_age_mort)
    mortality_rates = abs_mort[safe_age, partition.loc[alive_mask, 'sex']]

    # Calculate probability of death
    # Random numbers for each person
    random_draws = np.random.rand(len(mortality_rates))
    death_mask = random_draws < mortality_rates

    # Apply death to the 'alive' column for individuals whose random_draws are less than mortality_rates
    partition.loc[alive_mask, 'alive'] = ~death_mask

    return partition

def update_age_and_year(partition: pd.DataFrame, current_year: int) -> pd.DataFrame:
    """Updates age for alive individuals and sets current year."""
    if partition.empty:
        return partition
    partition.loc[partition['alive'], 'age'] += 1
    # You might want a 'current_sim_year' column if needed for other logic
    # partition['current_sim_year'] = current_year
    return partition

def apply_births(partition: pd.DataFrame, current_year: int, fertility_rates_df: pd.DataFrame, seed_offset: int, base_meta_df_for_dtypes: pd.DataFrame) -> pd.DataFrame:
    """
    Applies births to a population partition.
    This version vectorizes birth generation for the whole partition.
    """
    if partition.empty:
        return partition

    female_population = partition[(partition['sex'] == 1) & (partition['alive'] == True)].copy()

    if female_population.empty:
        return partition

    # Merge fertility rates based on age. Assume fertility_rates_df has 'age' and 'rate' columns
    # Ensure 'age' is integer for merging/joining
    female_population['age_int'] = female_population['age'].astype(int)
    merged_pop = pd.merge(female_population, fertility_rates_df[['age', 'rate']],
                          left_on='age_int', right_on='age', how='left', suffixes=('_pop', '_fert'))
    merged_pop['rate'] = merged_pop['rate'].fillna(0) # Fill NaN rates with 0

    # Calculate expected number of births per woman
    merged_pop['expected_births'] = merged_pop['rate']

    # Sample actual number of births using Poisson distribution
    # Removed 'time.time_ns()' as 'seed_offset' and hash of index should be sufficient for uniqueness
    local_random_state = np.random.RandomState(seed_offset + hash(frozenset(partition.index.values.tobytes())))
    num_births_per_woman = local_random_state.poisson(merged_pop['expected_births'])

    # Create new births DataFrame
    total_new_births = num_births_per_woman.sum()
    if total_new_births > 0:
        new_born_ids = np.arange(partition['person_id'].max() + 1, partition['person_id'].max() + 1 + total_new_births)
        new_born_sexes = local_random_state.randint(0, 2, total_new_births) # 0: Male, 1: Female

        new_born_df = pd.DataFrame({
            'person_id': new_born_ids,
            'sex': new_born_sexes,
            'age': 0,
            'year_born': current_year,
            'alive': True,
            'LTC': False,
            'QALY': 0.0,
            'cost': 0.0,
            'school': -1 # Babies are pre-school age
        })

        # Add any other required columns, ensuring dtypes match the main population DataFrame
        # Use base_meta_df_for_dtypes to get correct dtypes for new columns
        for col in base_meta_df_for_dtypes.columns.drop(new_born_df.columns, errors='ignore'):
            col_dtype = base_meta_df_for_dtypes[col].dtype
            if pd.api.types.is_numeric_dtype(col_dtype):
                new_born_df[col] = 0
            elif pd.api.types.is_bool_dtype(col_dtype):
                new_born_df[col] = False
            else:
                new_born_df[col] = base_meta_df_for_dtypes[col].iloc[0] # Fallback to dummy row value
            new_born_df[col] = new_born_df[col].astype(col_dtype)


        partition = pd.concat([partition, new_born_df], ignore_index=True)

    return partition

def update_qaly_cost(partition: pd.DataFrame, current_year: int, life_exp_np: np.ndarray, emp_np: np.ndarray, morb_keys: Dict[str, Any]) -> pd.DataFrame:
    """Updates QALY and cost for the population based on current year's attributes."""
    if partition.empty:
        return partition

    # Only update for alive individuals
    alive_mask = partition['alive']
    alive_df = partition[alive_mask]

    if alive_df.empty:
        return partition

    # Example: QALY calculation (simplified)
    # Assume QALY depends on age and health state (LTC)
    # For demonstration, let's use a simplified age-based QALY decrement and LTC decrement
    base_qaly = 1.0 # Max QALY per year

    # Age-dependent QALY: For simplicity, let's say QALY decreases slightly with age
    # Ensure age is within bounds for life_exp_np if used for QALY
    max_age_qaly = life_exp_np.shape[0] - 1
    safe_age_qaly = np.clip(alive_df['age'], 0, max_age_qaly)
    # Using life_exp_np as a proxy for QALY reduction factor (e.g., higher life_exp, higher QALY)
    # This is a placeholder; actual QALY calculation would be more complex.
    qaly_reduction_factor = life_exp_np[safe_age_qaly, alive_df['sex']] / 80.0 # Normalize to something reasonable
    qaly_reduction_factor = np.clip(qaly_reduction_factor, 0.5, 1.0) # Keep between 0.5 and 1.0

    current_qaly = base_qaly * qaly_reduction_factor

    # LTC effect on QALY: If LTC is True, reduce QALY
    current_qaly = np.where(alive_df['LTC'], current_qaly * 0.7, current_qaly) # 30% QALY reduction for LTC

    # Ensure current_qaly has the same index as alive_df before assignment
    partition.loc[alive_mask, 'QALY'] += pd.Series(current_qaly, index=alive_df.index) # Accumulate QALYs annually

    # Example: Cost calculation (simplified)
    # Assume cost depends on age, LTC, and employment status (simplified to 'school' for now)
    base_cost_per_year = 1000 # Base healthcare cost

    # Age-dependent cost: Costs increase with age
    # Using emp_np as a proxy for cost factor (e.g., higher emp rate implies healthier, lower cost?)
    # This is a placeholder; actual cost calculation would be more complex.
    max_age_cost = emp_np.shape[0] - 1
    safe_age_cost = np.clip(alive_df['age'], 0, max_age_cost)
    cost_factor = 1.0 + (alive_df['age'] / 100.0) # Simple linear increase with age

    # LTC effect on cost: If LTC is True, increase cost
    cost_factor = np.where(alive_df['LTC'], cost_factor * 2.0, cost_factor) # Double cost for LTC

    # Employment status effect (using 'school' as proxy for 'employed' or 'unemployed' states)
    # For example, if school == 1 (adult), assume higher engagement/cost
    # This is highly simplified and depends on your model's logic.
    current_cost = np.where(alive_df['school'] == 1, current_cost * 1.1, current_cost) # Adult population might have higher costs

    # Ensure current_cost has the same index as alive_df before assignment
    partition.loc[alive_mask, 'cost'] += pd.Series(current_cost, index=alive_df.index) # Accumulate costs annually

    return partition

def update_health_states(partition: pd.DataFrame, current_year: int, morb_keys: Dict[str, Any]) -> pd.DataFrame:
    """Updates health states (e.g., LTC status) for the population."""
    if partition.empty:
        return partition

    # Example: Simple probabilistic transition to LTC based on age
    # Placeholder: Assuming a base probability of developing LTC increases with age
    # This would typically involve specific transition matrices or rules from `morb_keys`

    # Only apply to alive individuals who don't already have LTC
    eligible_mask = (partition['alive'] == True) & (partition['LTC'] == False)
    eligible_df = partition[eligible_mask]

    if eligible_df.empty:
        return partition

    # Simplified probability of acquiring LTC
    # Let's say it's 0.001 + age * 0.0001
    prob_acquire_ltc = 0.001 + (eligible_df['age'] * morb_keys['dummy_morbidity_param_factor'])
    prob_acquire_ltc = np.clip(prob_acquire_ltc, 0, 0.1) # Cap max probability at 10%

    # Randomly determine if LTC is acquired
    random_draws = np.random.rand(len(eligible_df))
    acquire_ltc_mask = random_draws < prob_acquire_ltc

    partition.loc[eligible_mask, 'LTC'] = acquire_ltc_mask

    return partition

# --- Population System Class (Dask-enabled) ---

class PopulationSystemDask:
    def __init__(self, initial_pop_size: int, params: Dict[str, Any], scenario_type: str,
                 seed_offset: int, n_partitions: int, persist_dir: str = None):
        # Retrieve client within the worker context for this class's methods
        client = get_client()

        self.params = params
        self.scenario_type = scenario_type
        self.n_partitions = n_partitions
        # Removed 'time.time_ns()' as 'os.getpid()' and 'seed_offset' are sufficient for uniqueness
        self.persist_dir = persist_dir if persist_dir else f'dask-persist-{os.getpid()}-{seed_offset}'
        self.seed_offset = seed_offset

        # Retrieve published datasets from within the worker
        self.abs_mort = client.get_dataset("abs_mort_dataset")
        self.life_exp_np = client.get_dataset("life_exp_np_dataset")
        self.emp_np = client.get_dataset("emp_np_dataset")
        self.morb_keys = client.get_dataset("morb_keys_dataset")
        self.age_dist = client.get_dataset("age_dist_dataset")
        self.fertility_rates = client.get_dataset("fertility_rates_dataset")
        self.base_meta_df_for_dtypes = client.get_dataset("base_meta_df_for_dtypes_dataset")

        # The Dask metadata for the population DataFrame
        # This is derived from the base_meta_df_for_dtypes
        self.pop_meta = self.base_meta_df_for_dtypes.head(0)

        # Initial population generation
        initial_pop_dd = dd.from_delayed(
            client.submit(gen_population_raw_pandas, initial_pop_size, START_YEAR, self.seed_offset,
                          self.age_dist, self.base_meta_df_for_dtypes),
            meta=self.pop_meta
        ).repartition(npartitions=self.n_partitions)
        self.pop = initial_pop_dd.persist()

        # Setup persistence directory
        if self.persist_dir:
            os.makedirs(self.persist_dir, exist_ok=True)

    def run_year(self, year: int):
        """Runs the simulation for a single year."""
        # Retrieve client within the worker context for this method
        client = get_client() # Still good practice to explicitly get it for clarity, even if not directly used for client.rebalance

        # 1. Update Age
        self.pop = self.pop.map_partitions(update_age_and_year, current_year=year, meta=self.pop_meta)
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

        # 2. Apply Mortality
        self.pop = self.pop.map_partitions(
            apply_mortality,
            abs_mort=self.abs_mort,
            current_year=year,
            meta=self.pop_meta
        )
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

        # 3. Apply Births
        self.pop = self.pop.map_partitions(
            apply_births,
            current_year=year,
            fertility_rates_df=self.fertility_rates,
            seed_offset=self.seed_offset + year * 1000,
            base_meta_df_for_dtypes=self.base_meta_df_for_dtypes,
            meta=self.pop_meta
        )
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

        # 4. Update QALY and Cost (for alive individuals)
        self.pop = self.pop.map_partitions(
            update_qaly_cost,
            current_year=year,
            life_exp_np=self.life_exp_np,
            emp_np=self.emp_np,
            morb_keys=self.morb_keys,
            meta=self.pop_meta
        )
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

        # 5. Update Health States (e.g., LTC for alive individuals)
        self.pop = self.pop.map_partitions(
            update_health_states,
            current_year=year,
            morb_keys=self.morb_keys,
            meta=self.pop_meta
        )
        self.pop = self.pop.persist()
        client.rebalance(self.pop)

    def get_results(self) -> Tuple[float, float]:
        """Calculates total costs and QALYs for the simulation."""
        # Ensure 'alive' column is boolean for filtering
        self.pop['alive'] = self.pop['alive'].astype(bool)

        total_cost = self.pop['cost'].sum().compute()
        total_qaly = self.pop['QALY'].sum().compute()
        return total_cost, total_qaly

    def cleanup(self):
        """Cleans up any temporary persistence directories."""
        # Removed the unused 'client = get_client()' line
        if self.persist_dir and os.path.exists(self.persist_dir):
            shutil.rmtree(self.persist_dir)


# --- Simulation Runner Function ---

def run_sim(pop_size: int, sampled_params: Dict[str, Any], seed_idx: int,
            n_partitions_per_sim: int) -> Tuple[float, float]:
    """
    Runs a single simulation comparing retrospective and prospective arms for total costs and QALYs.

    Args:
        pop_size (int): Initial population size for this simulation.
        sampled_params (Dict[str, Any]): Parameters for this specific simulation run.
        seed_idx (int): Base seed for this simulation run to ensure reproducibility.
        n_partitions_per_sim (int): Number of Dask partitions for this simulation's population.

    Returns:
        Tuple[float, float]: A tuple containing (incremental_cost, incremental_qaly).
    """
    # Get the worker-local client
    worker_client = get_client()

    # Create a temporary directory for each simulation run
    # Removed 'time.time_ns()' from temp_persist_dir as 'os.getpid()' and 'seed_idx' are sufficient
    temp_persist_dir = f'dask_sim_data_{os.getpid()}_{seed_idx}'

    # Initialize simulation systems for retrospective and prospective arms
    # All static global data (abs_mort, life_exp_np, etc.) are now retrieved within
    # PopulationSystemDask's __init__ method using get_client().get_dataset().
    # Thus, no need to pass them as arguments to PopulationSystemDask constructor here.
    sim_r = PopulationSystemDask(pop_size, sampled_params, 'retrospective',
                                 seed_offset=seed_idx, n_partitions=n_partitions_per_sim,
                                 persist_dir=os.path.join(temp_persist_dir, 'retrospective'))
    sim_p = PopulationSystemDask(pop_size, sampled_params, 'prospective',
                                 seed_offset=seed_idx + 100000, n_partitions=n_partitions_per_sim,
                                 persist_dir=os.path.join(temp_persist_dir, 'prospective'))

    # Run simulation for each year
    for year in range(START_YEAR, END_YEAR + 1):
        sim_r.run_year(year)
        sim_p.run_year(year)

    # Get results
    total_cost_r, total_qaly_r = sim_r.get_results()
    total_cost_p, total_qaly_p = sim_p.get_results()

    # Clean up persisted data using fire_and_forget with the worker_client
    fire_and_forget(worker_client.submit(sim_r.cleanup))
    fire_and_forget(worker_client.submit(sim_p.cleanup))
    # Also clean up the parent temporary directory
    fire_and_forget(worker_client.submit(shutil.rmtree, temp_persist_dir, ignore_errors=True)) # Added ignore_errors for robustness

    # Calculate incremental costs and QALYs
    incremental_cost = total_cost_p - total_cost_r
    incremental_qaly = total_qaly_p - total_qaly_r

    return incremental_cost, incremental_qaly


# --- Main Execution ---

if __name__ == "__main__":
    print(f"--- Running Module 6: Deterministic Scenarios ---")

    # --- 1. Define Global Static Data ---
    # These are placeholder arrays; replace with your actual data.
    ABS_MORT = np.random.rand(101, 2) * 0.05 # Age 0-100, 2 sexes
    LIFE_EXP_NP = np.random.rand(101, 2) * 80 # Age 0-100, 2 sexes
    EMP_NP = np.random.rand(101, 2) * 0.8 # Age 0-100, 2 sexes (employment rates)
    MORB_KEYS = {'dummy_morbidity_param_factor': 0.0001} # Placeholder

    AGE_DIST = np.random.normal(loc=30, scale=15, size=1000).astype(int)
    AGE_DIST = AGE_DIST[ (AGE_DIST >= 0) & (AGE_DIST <= 100) ] # Ensure ages are realistic
    AGE_DIST = np.unique(AGE_DIST) # Just unique ages for sampling, or use weighted distribution

    # Fertility rates: age-specific rates (example data)
    FERTILITY_RATES_DATA = {
        'age': np.arange(15, 50),
        'rate': np.linspace(0.005, 0.1, 35) # Example: rates increasing then decreasing
    }
    FERTILITY_RATES_DATA['rate'][17:25] *= 1.5 # Peak around 32-40
    FERTILITY_RATES_DATA['rate'][25:] *= np.linspace(1.0, 0.1, 10) # Decrease after peak

    FERTILITY_RATES_DF = pd.DataFrame(FERTILITY_RATES_DATA)

    # --- Create the base_meta_df_for_dtypes with a dummy row ---
    # This DataFrame will have a single row with default values for each column,
    # ensuring that .iloc[0] is always safe and provides the correct dtype.
    meta_cols_definition = {
        'person_id': 'int64',
        'sex': 'int64',
        'age': 'int64',
        'year_born': 'int64',
        'alive': 'bool',
        'LTC': 'bool',
        'QALY': 'float64',
        'cost': 'float64',
        'school': 'int64',
        # Add any other columns your simulation might introduce
        # 'health_status': 'category',
        # 'region': 'int64',
    }
    dummy_data_for_meta = {
        col: pd.NA if dtype == 'object' else (0 if 'int' in dtype or 'float' in dtype else False)
        for col, dtype in meta_cols_definition.items()
    }
    # Create the DataFrame with one dummy row and ensure correct dtypes
    BASE_META_DF_FOR_DTYPES = pd.DataFrame([dummy_data_for_meta]).astype(meta_cols_definition)
    print("Base metadata DataFrame for dtypes created with a dummy row.")


    # --- Publish all static data as datasets ---
    # These will be retrieved directly by workers using client.get_dataset()
    client.publish_dataset(abs_mort_dataset=ABS_MORT)
    client.publish_dataset(life_exp_np_dataset=LIFE_EXP_NP)
    client.publish_dataset(emp_np_dataset=EMP_NP)
    client.publish_dataset(morb_keys_dataset=MORB_KEYS)
    client.publish_dataset(age_dist_dataset=AGE_DIST)
    client.publish_dataset(fertility_rates_dataset=FERTILITY_RATES_DF)
    client.publish_dataset(base_meta_df_for_dtypes_dataset=BASE_META_DF_FOR_DTYPES)

    print("All global parameters published to Dask cluster as datasets.")

    # --- 2. Define Simulation Parameters for each run ---
    params = []
    pop_sizes = []
    seeds = []
    parts = []

    for i in range(ITERATIONS):
        params.append({'param_a': i * 0.1, 'param_b': 100 + i * 5})
        pop_sizes.append(100_000) # Initial population size for each run
        seeds.append(i * 1000) # Unique seed for each run
        parts.append(4) # Number of partitions for the Dask DataFrame in each simulation

    # --- 3. Run Simulations in Parallel ---
    # `client.map` now only receives iterables that vary per task.
    # All static global data is fetched inside `run_sim` using `get_client().get_dataset()`.
    # The `pure=False` is essential because `run_sim` interacts with the Dask cluster
    # (via `get_client().get_dataset()` and `fire_and_forget(get_client().submit(...))`).
    futures = client.map(run_sim,
                         pop_sizes,
                         params,
                         seeds,
                         parts,
                         pure=False
                        )

    # Use TqdmCallback to show progress
    with TqdmCallback(total=len(futures), desc="Deterministic Scenarios") as cb:
        deterministic_results_raw = client.gather(futures)

    deterministic_results: List[Dict[str, Any]] = []
    for i, (dc, dq) in enumerate(deterministic_results_raw):
        result = {
            'iteration': i,
            'param_a': params[i]['param_a'],
            'param_b': params[i]['param_b'],
            'incremental_cost': dc,
            'incremental_qaly': dq
        }
        deterministic_results.append(result)

    print("\n--- Simulation Results ---")
    for res in deterministic_results:
        print(f"Iteration {res['iteration']}: Cost Diff={res['incremental_cost']:.2f}, QALY Diff={res['incremental_qaly']:.2f}")

    # --- Shutdown Dask Client and Cluster ---
    client.close()
    cluster.close()
    print("\nDask client and cluster shut down.")